In [1]:
!pip install tensorflow opencv-python numpy pillow scikit-learn openpyxl

In [2]:
import os, random, json, csv
import numpy as np
import cv2
from PIL import Image
from datetime import datetime
import openpyxl

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
STUDENTS_FOLDER = 'Faces'
IMG_SIZE        = (128, 128)
SEED            = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

In [4]:
def load_img(path):
    img = Image.open(path).convert('RGB').resize(IMG_SIZE)
    return np.array(img).astype(np.float32) / 255.0

X      = []
labels = []

for filename in os.listdir(STUDENTS_FOLDER):
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        path = os.path.join(STUDENTS_FOLDER, filename)
        name = os.path.splitext(filename)[0]
        name = '_'.join(name.split('_')[:-1])
        X.append(load_img(path))
        labels.append(name)

X = np.array(X)
print('Dataset shape:', X.shape)

Dataset shape: (2377, 128, 128, 3)


In [5]:
le = LabelEncoder()
y  = le.fit_transform(labels)

In [6]:
le      = LabelEncoder()
y       = le.fit_transform(labels)
n_class = len(le.classes_)

print('Number of students:', n_class)

Number of students: 29


In [7]:
print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (2377, 128, 128, 3)
y shape: (2377,)


In [8]:
le.classes_

array(['Alia Bhatt', 'Amitabh Bachchan', 'Andy Samberg', 'Anushka Sharma',
       'Billie Eilish', 'Brad Pitt', 'Camila Cabello', 'Charlize Theron',
       'Claire Holt', 'Courtney Cox', 'Dwayne Johnson', 'Elizabeth Olsen',
       'Ellen Degeneres', 'Henry Cavill', 'Hrithik Roshan',
       'Hugh Jackman', 'Jessica Alba', 'Kashyap', 'Lisa Kudrow',
       'Margot Robbie', 'Marmik', 'Natalie Portman', 'Priyanka Chopra',
       'Robert Downey Jr', 'Roger Federer', 'Tom Cruise',
       'Vijay Deverakonda', 'Virat Kohli', 'Zac Efron'], dtype='<U17')

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=SEED
)

print('Train:', len(X_train))
print('Val  :', len(X_val))
print('Test :', len(X_test))

Train: 1717
Val  : 303
Test : 357


In [10]:
n_class = len(le.classes_)
print('n_class:', n_class)

n_class: 29


In [11]:
inp = Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = Conv2D(32, 3, activation='relu', padding='same')(inp)
x = MaxPooling2D()(x)
x = Conv2D(64, 3, activation='relu', padding='same')(x)
x = MaxPooling2D()(x)
x = Conv2D(128, 3, activation='relu', padding='same')(x)

In [ ]:
x = MaxPooling2D()(x)
x = Flatten()(x)
x = Dropout(0.3)(x)
x   = Dense(256, activation='relu')(x)  
x   = Dropout(0.2)(x)                   
out = Dense(n_class, activation='softmax')(x)

In [13]:
X

array([[[[0.        , 0.05490196, 0.05490196],
         [0.        , 0.05490196, 0.05490196],
         [0.        , 0.05490196, 0.05490196],
         ...,
         [0.6666667 , 0.6745098 , 0.6627451 ],
         [0.6666667 , 0.6745098 , 0.6627451 ],
         [0.6666667 , 0.6745098 , 0.6627451 ]],

        [[0.        , 0.05490196, 0.05490196],
         [0.        , 0.05490196, 0.05490196],
         [0.        , 0.05490196, 0.05490196],
         ...,
         [0.6666667 , 0.6745098 , 0.6627451 ],
         [0.6666667 , 0.6745098 , 0.6627451 ],
         [0.6666667 , 0.6745098 , 0.6627451 ]],

        [[0.        , 0.05490196, 0.05490196],
         [0.        , 0.05490196, 0.05490196],
         [0.        , 0.05490196, 0.05490196],
         ...,
         [0.6666667 , 0.6745098 , 0.6627451 ],
         [0.6666667 , 0.6745098 , 0.6627451 ],
         [0.6666667 , 0.6745098 , 0.6627451 ]],

        ...,

        [[0.14509805, 0.11764706, 0.08627451],
         [0.15686275, 0.13333334, 0.10196079]

In [14]:
model = Model(inp, out)
model.compile(
    optimizer = 'adam',
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     8,388,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 29)             │         7,453 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,489,565 (32.39 MB)

 Trainable params: 8,489,565 (32.39 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


early_stop = EarlyStopping(
    monitor  = 'val_accuracy',
    patience = 5,
    restore_best_weights = True, 
    verbose  = 1
)

reduce_lr = ReduceLROnPlateau(
    monitor  = 'val_accuracy',
    factor   = 0.5,   
    patience = 3,
    min_lr   = 1e-6,
    verbose  = 1
)

history = model.fit(
    X_train, y_train,
    epochs          = 100,       
    batch_size      = 8,
    validation_data = (X_val, y_val),
    callbacks       = [early_stop, reduce_lr]
)

Epoch 1/100
215/215 ━━━━━━━━━━━━━━━━━━━━ 24s 102ms/step - accuracy: 0.1206 - loss: 3.0969 - val_accuracy: 0.2112 - val_loss: 2.8134 - learning_rate: 0.0010
Epoch 2/100
215/215 ━━━━━━━━━━━━━━━━━━━━ 21s 98ms/step - accuracy: 0.3774 - loss: 2.1091 - val_accuracy: 0.4818 - val_loss: 1.8684 - learning_rate: 0.0010
Epoch 3/100
215/215 ━━━━━━━━━━━━━━━━━━━━ 21s 97ms/step - accuracy: 0.5719 - loss: 1.4271 - val_accuracy: 0.5413 - val_loss: 1.4705 - learning_rate: 0.0010
Epoch 4/100
215/215 ━━━━━━━━━━━━━━━━━━━━ 21s 98ms/step - accuracy: 0.7199 - loss: 0.9261 - val_accuracy: 0.5908 - val_loss: 1.4645 - learning_rate: 0.0010
Epoch 5/100
215/215 ━━━━━━━━━━━━━━━━━━━━ 21s 97ms/step - accuracy: 0.7833 - loss: 0.6427 - val_accuracy: 0.5974 - val_loss: 1.5042 - learning_rate: 0.0010
Epoch 6/100
215/215 ━━━━━━━━━━━━━━━━━━━━ 21s 97ms/step - accuracy: 0.8422 - loss: 0.4928 - val_accuracy: 0.6337 - val_loss: 1.5497 - learning_rate: 0.0010
Epoch 7/100
215/215 ━━━━━━━━━━━━━━━━━━━━ 21s 99ms/step - accuracy: 0.

In [24]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
loss, acc 

(1.8918144702911377, 0.6358543634414673)

In [17]:
y_pred = np.argmax(model.predict(X_test), axis=1)
print('\nReport:\n', classification_report(y_test, y_pred, target_names=le.classes_))

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step

Report:
                    precision    recall  f1-score   support

       Alia Bhatt       0.29      1.00      0.44         2
 Amitabh Bachchan       0.82      1.00      0.90        14
     Andy Samberg       0.50      0.70      0.58        10
   Anushka Sharma       0.58      0.47      0.52        15
    Billie Eilish       0.50      0.64      0.56        14
        Brad Pitt       0.61      0.50      0.55        22
   Camila Cabello       0.73      0.92      0.81        12
  Charlize Theron       0.89      0.40      0.55        20
      Claire Holt       0.31      0.50      0.38         8
     Courtney Cox       0.44      0.57      0.50         7
   Dwayne Johnson       0.86      0.60      0.71        10
  Elizabeth Olsen       0.50      0.43      0.46         7
  Ellen Degeneres       0.89      0.80      0.84        10
     Henry Cavill       0.55      0.71      0.62        17
   Hrithik Roshan       0.58      0.44      0.50        16
     

In [18]:
feature_extractor = Model(inputs=model.input, outputs=model.layers[-2].output)
feature_extractor

<Functional name=functional_1, built=True>

In [19]:
import pandas as pd

df = pd.read_excel(r'D:/Dev/datascience-course/final_project2/Students_Images/Students.xlsx')

student_features = {}

for _, row in df.iterrows():
    name     = str(row['Name'])
    img_path = str(row['Image Path'])

    if not os.path.exists(img_path):
        print(f'Image not found: {img_path}')
        continue

    img = load_img(img_path)
    img = np.expand_dims(img, axis=0)

    features = feature_extractor.predict(img, verbose=0).flatten()

    if name in student_features:
        student_features[name] = (student_features[name] + features) / 2
    else:
        student_features[name] = features

class_names = list(student_features.keys())
class_names

['Malak', 'ali']

In [20]:
import json

model.save('face_model.keras')

embeddings = {name: feat.tolist() for name, feat in student_features.items()}
with open('student_features.json', 'w') as f:
    json.dump(embeddings, f)

In [21]:
!pip install mediapipe --user

In [22]:
import cv2
import numpy as np

def detect_drowsiness(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    eye_cascade  = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')
    
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    
    if len(faces) == 0:
        return 'No Face'
    
    for (x, y, w, h) in faces:
        roi_gray = gray[y:y+h, x:x+w]
        eyes     = eye_cascade.detectMultiScale(roi_gray)
        
        if len(eyes) >= 2:
            return 'Active'
        else:
            return 'Sleepy'
    
    return 'No Face'



In [23]:
import json

model.save("face_model.keras")

embeddings = {name: feat.tolist() for name, feat in student_features.items()}
with open("student_features.json", "w") as f:
    json.dump(embeddings, f)
